In [ ]:
import pickle
import os
import random

# grammar_name = "pcfg_cfg3b_eq_len_uniform_prob"
grammar_name = "pcfg_cfg3b_eq_len_skewed_prob"
# grammar_name = "pcfg_cfg3b_eq_len_skewed_prob_0.75"
# grammar_name = "pcfg_cfg3b_eq_len_skewed_prob_0.90"
# grammar_name = "pcfg_4_3_1_2_3_4_5_6_7_8_9_eq_len_skewed_prob"


# grammar_name = "pcfg_4_3_1_2_3_4_5_6_7_8_9_eq_len_all_rules_skewed_prob"

# grammar_name = "preg_alphabet_combined"
# grammar_name = "preg_alphabet_combined_varying_freq"


In [ ]:
# manually modify frequency by combining several grammars
if grammar_name == "preg_alphabet_combined_varying_freq":
    modified_data = {}
    modified_data["train_sequences"] = []
    modified_data["test_sequences"] = []
    for component_grammar_name, multiplier in zip(["preg_alphabet_2", "preg_alphabet_7", "preg_alphabet_26"], 
                                                  [10, 5, 1]):

        ideal_training_size = 5000
        training_size_original = None
        if os.path.isfile(f"../data/{component_grammar_name}/sequences_w_edit_distance_{component_grammar_name}_{ideal_training_size * 2}_5.pkl"):
            with open(f"../data/{component_grammar_name}/sequences_w_edit_distance_{component_grammar_name}_10000_5.pkl", 'rb') as f:
                data = pickle.load(f)
                assert "train_sequences" in data
                training_size_original = len(data["train_sequences"])
                for key in data:
                    print(key, len(data[key]), len(set(data[key])))
                    for i, sequence in enumerate(data[key]):
                        assert sequence == data[key][i]

                # only consider training sequences
                for _ in range(multiplier):
                    modified_data["train_sequences"] += data["train_sequences"]
                    modified_data["test_sequences"] += data["test_sequences"]
        
        print(multiplier)
        print(training_size_original)
    print(len(modified_data["train_sequences"]))
    os.system(f"mkdir -p ../data/{grammar_name}")
    with open(f"../data/{grammar_name}/sequences_w_edit_distance_{grammar_name}_10000_5.pkl", 'wb') as f:
        pickle.dump(modified_data, f)

In [ ]:
ideal_training_size = 5000
training_size_original = None
if os.path.isfile(f"../data/{grammar_name}/sequences_w_edit_distance_{grammar_name}_{ideal_training_size * 2}_5.pkl"):
    with open(f"../data/{grammar_name}/sequences_w_edit_distance_{grammar_name}_10000_5.pkl", 'rb') as f:
        data = pickle.load(f)
        assert "train_sequences" in data
        training_size_original = len(data["train_sequences"])
        for key in data:
            print(key, len(data[key]), len(set(data[key])))
            for i, sequence in enumerate(data[key]):
                assert sequence == data[key][i]

training_size_original

if grammar_name == "preg_alphabet_combined_varying_freq":
    ideal_training_size = 80000 * 6

In [ ]:
train_sequence_freq = {}
for sequence in data["train_sequences"]:
    if sequence not in train_sequence_freq:
        train_sequence_freq[sequence] = 0
    train_sequence_freq[sequence] += 1

# # sort by frequency
# train_sequence_freq = sorted(train_sequence_freq.items(), key=lambda x: x[1], reverse=True)

# sort by length
train_sequence_freq = sorted(train_sequence_freq.items(), key=lambda x: len(x[0]), reverse=True)


for i, (sequence, frequency) in enumerate(train_sequence_freq):
    print(sequence[:20], frequency, i)

In [ ]:
random.seed(5)

num_counterfactual_examples = 3
counterfactual_examples = []
frequence_counterfactual_examples = []


# for i in range(num_counterfactual_examples):
#     string = data["train_sequences"][random.randint(0, len(data["train_sequences"]) - 1)]
#     if string not in counterfactual_examples:
#         freq_string = 0
#         for sequence in data["train_sequences"]:
#             if sequence == string:
#                 freq_string += 1
#         counterfactual_examples.append(string)
#         frequence_counterfactual_examples.append(int(freq_string*(ideal_training_size/training_size_original)))


# for i in [0, 1, -1]:
# for i in [1, 13, 3]: # for alphabet_combined grammar where we select three strings of different information content.
for i in [8, 0, 1]: # for preg_alphabet_combined_varying_freq grammar where we select three strings of different information content.
    string = train_sequence_freq[i][0]
    if string not in counterfactual_examples:
        freq_string = 0
        for sequence in data["train_sequences"]:
            if sequence == string:
                freq_string += 1
        counterfactual_examples.append(string)
        frequence_counterfactual_examples.append(max(int(freq_string*(ideal_training_size/training_size_original)), 1))



assert len(counterfactual_examples) == num_counterfactual_examples
assert len(frequence_counterfactual_examples) == num_counterfactual_examples

for string, frequency in zip(counterfactual_examples, frequence_counterfactual_examples):
    print(string[:20], frequency)

In [ ]:
# training sequences without these three examples
train_sequences_exclude = []
for sequence in data["train_sequences"]:
    if sequence not in counterfactual_examples:
        train_sequences_exclude.append(sequence)

for string in counterfactual_examples:
    assert string not in train_sequences_exclude
    
len(train_sequences_exclude)

In [ ]:
data_modified = {}
for key in data:
    if key == "train_sequences":
        data_modified[key] = train_sequences_exclude
    else:
        data_modified[key] = data[key]

for i, string in enumerate(counterfactual_examples):
    data_modified[f"counterfactual_{i}"] = [string for _ in range(frequence_counterfactual_examples[i])]

for key in data_modified:
    print(key, len(data_modified[key]), len(set(data_modified[key])))

In [ ]:
grammar_name_modified = grammar_name + "_counterfactual"
os.makedirs(f"../data/{grammar_name_modified}", exist_ok=True)
with open(f"../data/{grammar_name_modified}/sequences_w_edit_distance_{grammar_name_modified}_10000_5.pkl", 'wb') as f:
    pickle.dump(data_modified, f)
    print(f"Saved modified data to ../data/{grammar_name_modified}/sequences_w_edit_distance_{grammar_name_modified}_10000_5.pkl")